# Chapter 2: Attention, Transformer and GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch02_transformer_gpt.ipynb)

This notebook is generated from the complete chapter source. Nothing is replaced by a toy implementation. Python files are only split into notebook cells for readability; concatenating those cells reproduces the original source exactly.

**Pinned upstream commit:** `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`


## Notebook architecture

The notebook follows the chapter as a readable pipeline rather than hiding implementation behind `%run` calls. Shared local modules used by the chapter are written from visible cells first, then every chapter script is presented in source order. Original model dimensions, algorithms, and training hyperparameters are preserved.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('working directory:', Path.cwd())
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch check:', exc)


## Shared modules used by this chapter

These cells keep shared architecture visible while preserving the original package layout for imports.


### `codebot/model.py`

SHA-256: `be6317fe93bae18715dcb68cd80790bb93350961df65ab53869488b38ef85476`


In [ ]:
%%writefile codebot/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, dropout_rate=0.1):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.attention_dropout = nn.Dropout(dropout_rate)
        self.output_dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)  # (B, C, H*D)
        K = self.W_k(x)  # (B, C, H*D)
        V = self.W_v(x)  # (B, C, H*D)

        Q = Q.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)
        K = K.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)
        V = V.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)

        scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, H, C, C)
        scores = scores / (D ** 0.5)

        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)  # (B, H, C, C)
        weights = self.attention_dropout(weights)
        hidden = torch.matmul(weights, V)  # (B, H, C, D)

        hidden = hidden.transpose(1, 2).contiguous()  # (B, C, H, D)
        hidden = hidden.view(B, C, H * D)  # (B, C, H*D)
        output = self.W_o(hidden)  # (B, C, E)
        output = self.output_dropout(output)

        return output


class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm_x + self.beta


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FFN(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),  # GELU()
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.layers(x)


class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, dropout_rate=0.1):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.LayerNorm(embed_dim)  # LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, dropout_rate)
        self.norm2 = nn.LayerNorm(embed_dim)  # LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, ff_dim, dropout_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, dropout_rate):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_context_len, embed_dim)
        self.dropout = nn.Dropout(dropout_rate)

        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, dropout_rate)
            for _ in range(n_layer)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size)

        self.embed.weight = self.unembed.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, C = ids.shape
        device = ids.device

        pos = torch.arange(0, C, dtype=torch.long, device=device)
        emb = self.embed(ids)
        pos_emb = self.pos_embed(pos)
        x = self.dropout(emb + pos_emb)

        for block in self.blocks:
            x = block(x)
        x = self.norm(x)

        logits = self.unembed(x)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            dropout_rate=checkpoint['dropout_rate']
        )

        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model


## Complete chapter source


## `ch02/01_soft_dict.py`

SHA-256: `71c1e6af48ada21cdb9c6d1e50d6342194d492bc99adcd25f65cd164cd6b174c`


**Imports**


In [ ]:
import numpy as np

# ディクショナリの例


**Execution**


In [ ]:
d = {
    'apple': 100,
    'banana': 200,
    'cherry': 300,
    'durian': 400
}

# キーを指定して値を取得
query = 'banana'
print(d[query])  # 200


# キー: (アクション性, ドラマ性, コメディ性) 各0〜10で表現
# バリュー: ユーザーの評価点 (0〜100点)
movie_preferences = {
    (8, 2, 3): 85,  # アクション重視の映画
    (3, 9, 1): 70,  # ドラマ重視の映画
    (1, 2, 9): 60,  # コメディ重視の映画
    (5, 5, 5): 75,  # バランスの取れた映画
    (7, 6, 2): 80,  # アクションドラマ
    (2, 7, 6): 65,  # コメディドラマ
    (9, 1, 1): 90,  # 純粋なアクション
}

# 新しい映画
new_movie = (6, 4, 5)



**Definitions**


In [ ]:
def soft_dictionary(query, dictionary):
    # 類似度
    similarity = []
    for key in dictionary:
        s = np.dot(query, key)
        similarity.append(s)

    # ソフトマックス
    exp_similarity = np.exp(similarity)
    weights = exp_similarity / np.sum(exp_similarity)

    # 重み付き和
    result = 0
    for weight, value in zip(weights, dictionary.values()):
        result += weight * value

    return result, weights




**Execution**


In [ ]:
predicted_rating, weights = soft_dictionary(new_movie, movie_preferences)

print(f"新しい映画 {new_movie} の予測評価: {predicted_rating:.2f} 点")
print("\n各映画の重み:")
for key, weight in zip(movie_preferences.keys(), weights):
    print(f"映画 {key}: {weight*100:.2f}%")


## `ch02/02_attn_math.py`

SHA-256: `54c253eaf5dd28a5690f78fda10b6bfbea33140b8d6cff0517c476cfe15a3585`


**Imports**


In [ ]:
import torch
import torch.nn.functional as F

# キー（映画のジャンル特性）


**Execution**


In [ ]:
K = torch.tensor([
    [8, 2, 3],  # アクション重視の映画
    [3, 9, 1],  # ドラマ重視の映画
    [1, 2, 9],  # コメディ重視の映画
    [5, 5, 5],  # バランスの取れた映画
    [7, 6, 2],  # アクションドラマ
    [2, 7, 6],  # コメディドラマ
    [9, 1, 1],  # 純粋なアクション
], dtype=torch.float32)

# バリュー（ユーザーの評価）
V = torch.tensor([
    [85],
    [70],
    [60],
    [75],
    [80],
    [65],
    [90]
], dtype=torch.float32)

# 新しい映画のジャンル特性（複数のクエリ）
Q = torch.tensor([
    [6, 4, 5],  # バランスの取れたアクション寄りの映画
    [2, 8, 3],  # ドラマ重視の映画
    [4, 3, 7],  # コメディ寄りの映画
], dtype=torch.float32)



**Definitions**


In [ ]:
def attention(Q, K, V):
    similarity = torch.matmul(Q, K.t())     # QK^Tを計算
    weights = F.softmax(similarity, dim=1)  # ソフトマックス関数
    output = torch.matmul(weights, V)       # 重み付き和
    return output, weights



**Execution**


In [ ]:
predicted_ratings, weights = attention(Q, K, V)

# 結果の表示
for movie, rating in zip(Q, predicted_ratings):
    print(f"映画 {movie.numpy()} の予測評価: {rating.item():.2f}")


## `ch02/03_attn_scaling.py`

SHA-256: `f643c71ada0944b235ee7d1da14f600e63e15057ddd1953739ad2eca5e7da6d1`


**Imports**


In [ ]:
import torch
import torch.nn.functional as F



**Execution**


In [ ]:
x = torch.tensor([100.0, 200.0, 300.0])
y = F.softmax(x, dim=0)
print(y)




**Imports**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt



**Execution**


In [ ]:
d = 10

q = np.random.randn(d)
k = np.random.randn(d)

dot_product = np.dot(q, k)
scaled_dot_product = dot_product / np.sqrt(d)

print('dot product:', dot_product)
print('scaled dot product:', scaled_dot_product)


d = 10
num_samples = 10000  # 先ほどの内積の計算を10000回行う

dot_products = []
scaled_dot_products = []

for _ in range(num_samples):
    q = np.random.randn(d)
    k = np.random.randn(d)

    dot_product = np.dot(q, k)
    scaled_dot_product = dot_product / np.sqrt(d)

    dot_products.append(dot_product)
    scaled_dot_products.append(scaled_dot_product)


# 結果をプロット
plt.figure(figsize=(10, 6))
plt.hist(dot_products, bins=50, alpha=0.5, label='Without scaling')
plt.hist(scaled_dot_products, bins=50, alpha=0.5, label='With scaling')
plt.legend()
plt.show()

print("Variances without scaling:", np.var(dot_products))
print("Variances with scaling:", np.var(scaled_dot_products))


## `ch02/06_attn_mask.py`

SHA-256: `988a358b1b49ef1ebf85f6dae791b3fee07805bc5287f72c460e8ef97d393d01`


**Imports**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F



**Definitions**


In [ ]:
class Attention(nn.Module):
    def __init__(self, embed_dim, key_dim):
        super().__init__()
        # Q, K, Vの変換行列
        self.W_q = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)

        self.key_dim = key_dim

    def forward(self, x):  # x: (B, C, E)
        Q = self.W_q(x)    # Q: (B, C, D)
        K = self.W_k(x)    # K: (B, C, D)
        V = self.W_v(x)    # V: (B, C, E)

        # Attentionマップの計算
        K_t = K.transpose(-2, -1)  # (B, D, C)
        scores = torch.matmul(Q, K_t)  # (B, C, C)
        scores = scores / (self.key_dim ** 0.5)

        # マスクの適用
        B, C, E = x.shape
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)  # (B, C, E)
        return output



**Execution**


In [ ]:
attention = Attention(embed_dim=256, key_dim=64)
x = torch.randn(2, 5, 256)  # (batch_size=2, context_len=5, embed_dim=256)
y = attention(x)

print("入力形状:", x.shape)
print("出力形状:", y.shape)


## `ch02/07_attn_value.py`

SHA-256: `fc3c1ec19393e209927222ed0ce37f3b603200c6d8b86a56eead6ab0ea30154d`


**Imports**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F



**Definitions**


In [ ]:
class Attention(nn.Module):
    def __init__(self, embed_dim, key_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_o = nn.Linear(key_dim, embed_dim, bias=False)  # 出力変換行列
        self.key_dim = key_dim

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        K_t = K.transpose(-2, -1)
        scores = torch.matmul(Q, K_t)
        scores = scores / (self.key_dim ** 0.5)

        B, C, E = x.shape
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)

        # 出力変換
        output = self.W_o(hidden)

        return output



**Execution**


In [ ]:
attention = Attention(embed_dim=256, key_dim=64)
x = torch.randn(2, 5, 256)
y = attention(x)

print("入力形状:", x.shape)
print("出力形状:", y.shape)


## `ch02/08_multi_head.py`

SHA-256: `231bb538cb963a0192a9df99ae1e7908892832352a2c5d72e6cf6c00cfc9f1e0`


**Imports**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F



**Execution**


In [ ]:
B = 2   # バッチサイズ（batch_size）
C = 4   # コンテキスト長（context_len）
E = 16  # 埋め込みの次元数（embed_dim）
H = 3   # ヘッド数（n_head）
D = 8   # ヘッドの次元数（head_dim）

# 入力テンソル
x = torch.randn(B, C, E)

# 効率的な実装：全ヘッド分の重みを一つの行列にまとめる
W_q = nn.Linear(E, H*D, bias=False)
W_k = nn.Linear(E, H*D, bias=False)
W_v = nn.Linear(E, H*D, bias=False)

Q = W_q(x)  # (B, C, H*D)
K = W_k(x)  # (B, C, H*D)
V = W_v(x)  # (B, C, H*D)

# 形状の変換
Q = Q.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)
K = K.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)
V = V.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)

scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, H, C, C)
scores = scores / (D ** 0.5)

# マスク処理
mask = torch.tril(torch.ones(C, C, device=scores.device))
scores = scores.masked_fill(mask == 0, float('-inf'))

# Attention重み
weights = F.softmax(scores, dim=-1)  # (B, H, C, C)
hidden = torch.matmul(weights, V)   # (B, H, C, D)

# 形状変換: (B, H, C, D) → (B, C, H*D)
hidden = hidden.transpose(1, 2)  # (B, C, H, D)
hidden = hidden.contiguous().view(B, C, H*D)  # (B, C, H*D)

# 出力変換: (B, C, H*D) → (B, C, E)
W_o = nn.Linear(H*D, E, bias=False)
output = W_o(hidden)  # (B, C, E)




**Definitions**


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, dropout_rate=0.1):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        # Dropoutを追加
        self.attention_dropout = nn.Dropout(dropout_rate)
        self.output_dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B, C, E = x.shape  # バッチサイズ、コンテキスト長、埋め込み次元
        H, D = self.n_head, self.head_dim  # ヘッドサイズ、ヘッドの次元数

        # Q, K, V の計算
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # 各ヘッドに分割して並べ替え
        Q = Q.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)
        K = K.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)
        V = V.view(B, C, H, D).transpose(1, 2)  # (B, H, C, D)

        scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, H, C, C)
        scores = scores / (D ** 0.5)

        # マスク処理
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        # Attention重み
        weights = F.softmax(scores, dim=-1)  # (B, H, C, C)
        weights = self.attention_dropout(weights)  # weightsにDropout
        hidden = torch.matmul(weights, V)  # (B, H, C, D)

        # ヘッドの結合と出力変換
        hidden = hidden.transpose(1, 2).contiguous()  # (B, C, H, D)
        hidden = hidden.view(B, C, H * D)  # (B, C, H*D)
        output = self.W_o(hidden)  # (B, C, E)
        output = self.output_dropout(output)  # 最終出力にDropout

        return output

# 使用例


**Execution**


In [ ]:
embed_dim = 512
n_head = 8
head_dim = 64

mha = MultiHeadAttention(embed_dim, n_head, head_dim)

# テスト用データ
batch_size = 2
context_len = 10
x = torch.randn(batch_size, context_len, embed_dim)

# 実行
output = mha(x)
print(f"入力形状: {x.shape}")       # (2, 10, 512)
print(f"出力形状: {output.shape}") # (2, 10, 512)


## `ch02/09_norm_gelu.py`

SHA-256: `363ef382b343cadfe262e6d0c20fa784818fba69ec51688aad0657ce40a57317`


**Imports**


In [ ]:
import os, sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')


**Imports**


In [ ]:
import torch
import torch.nn as nn
from codebot.model import MultiHeadAttention




**Definitions**


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm_x + self.beta


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FFN(nn.Module):
    def __init__(self, x_dim, hidden_dim=None, dropout_rate=0.1):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(4 * x_dim)

        self.layers = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            GELU(),
            nn.Linear(hidden_dim, x_dim),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.layers(x)


class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim=None, dropout_rate=0.1):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, dropout_rate)
        self.norm2 = LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, ff_dim, dropout_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


## `ch02/10_gpt2.py`

SHA-256: `7a9cafe8c8cb2a2f75fe20624f6de446bba1735b7efa7803bfba940a5e0a7b0f`


**Imports**


In [ ]:
import os, sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
import torch
import torch.nn as nn
from codebot.model import Block




**Definitions**


In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, dropout_rate):
        super().__init__()
        self.vocab_size = vocab_size            # 語彙サイズ
        self.max_context_len = max_context_len  # 最大コンテキスト長
        self.embed_dim = embed_dim              # 埋め込み次元数
        self.n_head = n_head                    # Attentionのヘッド数
        self.n_layer = n_layer                  # Transformerブロックの数
        self.ff_dim = ff_dim                    # FFNの隠れ層サイズ
        self.dropout_rate = dropout_rate        # ドロップアウト率

        # 埋め込み層
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_context_len, embed_dim)
        self.dropout = nn.Dropout(dropout_rate)

        # Transformerブロック
        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, dropout_rate)
            for _ in range(n_layer)
        ])

        # 出力層
        self.norm = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size)

        # 重み共有
        self.embed.weight = self.unembed.weight

        # 重みの初期化
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, C = ids.shape  # B: バッチサイズ、C: コンテキスト長
        device = ids.device

        # 埋め込み
        pos = torch.arange(0, C, dtype=torch.long, device=device)
        emb = self.embed(ids)
        pos_emb = self.pos_embed(pos)
        x = self.dropout(emb + pos_emb)

        # Transformerブロック
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)

        # 出力
        logits = self.unembed(x)  # (B, C, vocab_size)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            dropout_rate=checkpoint['dropout_rate']
        )
        # 重みの読み込み
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model




**Execution**


In [ ]:
vocab_size = 1000
max_context_len = 256
embed_dim = 384
n_head = 6
n_layer = 6
ff_dim = 4 * embed_dim
dropout_rate = 0.1

# モデルを作成
model = GPT(vocab_size, max_context_len, embed_dim, n_head,
             n_layer, ff_dim, dropout_rate)

# 動作テスト
dummy_input = torch.randint(0, vocab_size, (1, max_context_len))
logits = model(dummy_input)
print(f"出力形状: {logits.shape}")


## `ch02/graph.py`

SHA-256: `c45f0e0edd7ac8efb60ea91cff84e4cdb7f148d47a17ffd563230de0f04696c4`


**Imports**


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# データ生成


**Execution**


In [ ]:
x = np.linspace(-3, 3, 500)

# ReLU関数
relu = np.maximum(0, x)

# GELU関数（近似式）
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

# プロット作成
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(x, relu, 'b-', linewidth=2, label='ReLU')
ax.plot(x, gelu, 'r-', linewidth=2, label='GELU')

# 軸の設定
ax.set_xlim(-3, 3)
ax.set_ylim(-0.5, 3.0)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)

# グリッド
ax.grid(True, linestyle='--', alpha=0.7)
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)

# 凡例
ax.legend(loc='upper left', fontsize=12)

# 余白調整
plt.tight_layout()

# PNGで保存
plt.savefig('relu_gelu.png', format='png', bbox_inches='tight')
plt.close()


## T4 execution note

The implementation above keeps the upstream code and hyperparameters intact. For chapters with long training loops, a Colab T4 can execute the implementation, but completing the full training schedule may take substantial wall-clock time. No reduced model, shortened algorithm, or toy substitute is enabled by default in this notebook.
